In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "cryptohftdata"])
subprocess.run([sys.executable, "-m", "pip", "install", "polars"])

## Download all Files (Multi-Core)

In [ ]:
import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-10-01"
END_DATE = "2025-10-31"
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES
MAX_WORKERS = 6  # <- tune this (start low!)

DROP_HEADERS = [
    'some_unused_column', 'symbol',
    'transaction_time', 'event_time', 'timestamp', 'prev_final_update_id'
]


def optimize_floats(df):
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df


def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range(delta.days + 1)]


def process_single_date(date_str, category):
    """Runs inside each process."""
    client = chd.CryptoHFTDataClient(api_key=API_KEY)

    cat_name, fetch_method_name = category
    fetch_func = getattr(client, fetch_method_name)

    os.makedirs('orderbook_raw', exist_ok=True)
    file_path = f"orderbook_raw/{date_str}_{SYMBOL}.parquet"

    if os.path.exists(file_path):
        return f"[SKIP] {date_str}"

    try:
        df = fetch_func(
            symbol=SYMBOL,
            exchange=EXCHANGE,
            start_date=date_str,
            end_date=date_str
        )

        if df is None or df.empty:
            return f"[EMPTY] {date_str}"

        # Drop unused columns
        df.drop(columns=[c for c in DROP_HEADERS if c in df.columns],
                errors='ignore', inplace=True)

        # Convert time
        if 'received_time' in df.columns:
            df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

        # Optimize
        df = optimize_floats(df)

        # Save
        df.to_parquet(
            file_path,
            engine='pyarrow',
            compression='zstd',
            compression_level=9,
            index=False
        )

        rows = len(df)

        del df
        gc.collect()

        return f"[SAVED] {date_str} | Rows: {rows:,}"

    except Exception as e:
        if 'df' in locals():
            del df
        gc.collect()
        return f"[ERROR] {date_str}: {e}"


def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)

    categories = [
        ("orderbook", "get_orderbook"),
        #("trades", "get_trades"),
        #("open_interest", "get_open_interest"),
    ]

    for category in categories:
        cat_name, _ = category
        print(f"\n>>> Starting Category: {cat_name.upper()}")

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_date, date, category)
                for date in dates
            ]

            for future in as_completed(futures):
                print(" ", future.result())


if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")

## Now Split files into multiple segments:

In [ ]:
import polars as pl
from tqdm import tqdm
import os

def split_parquet_by_interval(input_file, num_intervals, prefix):
    # 1. Scan the file lazily
    lf = pl.scan_parquet(input_file)

    # 2. Get the start and end times
    # Since it's already datetime[ns], we just pull the min/max
    print("Analyzing file metadata...")
    time_bounds = lf.select([
        pl.col("received_time").min().alias("start"),
        pl.col("received_time").max().alias("end")
    ]).collect()

    start_time = time_bounds["start"][0]
    end_time = time_bounds["end"][0]

    if start_time is None or end_time is None:
        print("Error: Could not find time bounds. Is the 'received_time' column empty?")
        return

    total_duration = end_time - start_time
    interval_duration = total_duration / num_intervals

    print(f"Start: {start_time}")
    print(f"End:   {end_time}")
    print(f"Splitting into {num_intervals} chunks of ~{interval_duration} each.\n")

    # 3. Create output directory
    output_dir = "orderbook_chunks"
    os.makedirs(output_dir, exist_ok=True)

    # 4. Filter and save in a memory-efficient loop
    for i in tqdm(range(num_intervals), desc="Processing Chunks"):
        chunk_start = start_time + (i * interval_duration)
        chunk_end = start_time + ((i + 1) * interval_duration)

        # We use collect() inside the loop so only ONE slice is in RAM at a time
        chunk_df = (
            lf.filter(
                (pl.col("received_time") >= chunk_start) & 
                (pl.col("received_time") < chunk_end)
            )
            .collect()
        )

        if not chunk_df.is_empty():
            output_filename = f"{output_dir}/{prefix}_chunk_{i+1:02d}.parquet"
            chunk_df.write_parquet(output_filename)
        
        # Explicitly clear chunk from memory (optional, but good for very tight RAM)
        del chunk_df

    print(f"\nDone! Files are in the '{output_dir}' folder.")

import os

# Assuming split_parquet_by_interval is defined above...

if __name__ == "__main__":
    # Specify the folder containing your parquet files
    folder_path = "./orderbook_raw" 
    
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' not found.")
    else:
        # Get a list of all parquet files in the directory
        files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]
        
        if not files:
            print("No .parquet files found in the directory.")
        else:
            try:
                #intervals = int(input("Enter the number of intervals (e.g., 24 for hourly): "))
                intervals = 6
                
                for filename in files:
                    # Construct the full path to the file
                    file_path = os.path.join(folder_path, filename)
                    
                    print(f"--- Processing: {filename} ---")
                    split_parquet_by_interval(file_path, intervals, filename)
                
                print("\nAll files processed successfully.")
                
            except ValueError:
                print("Please enter a valid whole number.")

## Now reconstuct it?

## Reconstruction by Gemini:

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import json
from numba import njit
from datetime import datetime

# Configuration
INPUT_FOLDER = "orderbook_chunks"
OUTPUT_FOLDER = "features"
CHECKPOINT_FOLDER = "checkpoints"
OUTPUT_FILE = "orderbook_features.parquet"
INTERVAL_NS = 30 * 1_000_000_000  # 30 seconds
LEVELS_TO_EXTRACT = 10
SAVE_INTERVAL_FILES = 15  # Save every 15 subfiles

@njit
def compute_features_numba(bid_prices, bid_qtys, ask_prices, ask_qtys, timestamp):
    """
    Optimized feature computation using Numba.
    Expects sorted arrays (bids descending, asks ascending).
    """
    if len(bid_prices) == 0 or len(ask_prices) == 0:
        return np.zeros(20) # Return empty feature vector

    best_bid = bid_prices[0]
    best_bid_qty = bid_qtys[0]
    best_ask = ask_prices[0]
    best_ask_qty = ask_qtys[0]
    
    mid_price = (best_bid + best_ask) / 2.0
    spread = best_ask - best_bid
    spread_bps = (spread / mid_price) * 10000.0 if mid_price > 0 else 0.0
    micro_price = (best_bid * best_ask_qty + best_ask * best_bid_qty) / (best_bid_qty + best_ask_qty)

    # Output indices: 
    # 0: ts, 1: bb, 2: ba, 3: mid, 4: spread, 5: bps, 6: micro
    # 7-9: bid_vol (1,5,10), 10-12: ask_vol, 13-15: obi, 16: b_dens, 17: a_dens
    res = np.zeros(20)
    res[0] = timestamp
    res[1] = best_bid
    res[2] = best_ask
    res[3] = mid_price
    res[4] = spread
    res[5] = spread_bps
    res[6] = micro_price

    # Aggregated Volumes and OBI
    ns = np.array([1, 5, 10])
    for i in range(3):
        n = ns[i]
        b_vol = np.sum(bid_qtys[:n])
        a_vol = np.sum(ask_qtys[:n])
        res[7+i] = b_vol
        res[10+i] = a_vol
        
        total_v = b_vol + a_vol
        res[13+i] = (b_vol - a_vol) / total_v if total_v > 0 else 0.0

    # Densities
    bid_range = bid_prices[0] - bid_prices[min(len(bid_prices)-1, 9)]
    ask_range = ask_prices[min(len(ask_prices)-1, 9)] - ask_prices[0]
    
    res[16] = res[9] / bid_range if bid_range > 0 else 0.0
    res[17] = res[12] / ask_range if ask_range > 0 else 0.0

    return res

def get_sorted_book(book_dict, reverse=False):
    """Utility to convert dict to sorted numpy arrays."""
    if not book_dict:
        return np.array([]), np.array([])
    prices = np.sort(np.array(list(book_dict.keys())))
    if reverse:
        prices = prices[::-1]
    
    qtys = np.array([book_dict[p] for p in prices])
    return prices.astype(np.float64), qtys.astype(np.float64)

def load_progress():
    path = os.path.join(CHECKPOINT_FOLDER, "metadata.json")
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {"processed_files": [], "next_snapshot_time": None}

def save_progress(processed_files, next_snapshot_time, features_acc):
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)
    
    # Cast next_snapshot_time to standard Python int to avoid JSON serialization error
    serializable_time = int(next_snapshot_time) if next_snapshot_time is not None else None
    
    # Save Metadata
    with open(os.path.join(CHECKPOINT_FOLDER, "metadata.json"), 'w') as f:
        json.dump({"processed_files": processed_files, "next_snapshot_time": serializable_time}, f)
    
    # Save partial features
    if features_acc:
        cols = [
            'timestamp', 'best_bid', 'best_ask', 'mid_price', 'spread', 'spread_bps', 'micro_price',
            'bid_vol_1', 'bid_vol_5', 'bid_vol_10', 'ask_vol_1', 'ask_vol_5', 'ask_vol_10',
            'obi_1', 'obi_5', 'obi_10', 'bid_density', 'ask_density', 'empty1', 'empty2'
        ]
        # Convert list of numpy arrays to a DataFrame
        df = pd.DataFrame(np.array(features_acc), columns=cols)
        df.to_parquet(os.path.join(CHECKPOINT_FOLDER, f"partial_{len(processed_files)}.parquet"))

def process_orderbook_stream():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)
    
    progress = load_progress()
    processed_set = set(progress["processed_files"])
    next_snapshot_time = progress["next_snapshot_time"]

    bids = {} 
    asks = {}
    
    file_pattern = os.path.join(INPUT_FOLDER, "*.parquet")
    files = sorted(glob.glob(file_pattern))
    
    if not files:
        print("No input files found.")
        return

    all_features = []
    file_counter = 0
    
    print(f"Starting processing. Total files: {len(files)}")

    for file_path in files:
        if file_path in processed_set:
            continue

        print(f"[{datetime.now().strftime('%H:%M:%S')}] Processing: {os.path.basename(file_path)}")
        df = pd.read_parquet(file_path)

        if not pd.api.types.is_integer_dtype(df['received_time']):
            df['received_time'] = df['received_time'].values.astype('int64')

        times = df['received_time'].values
        sides = df['side'].values
        prices = df['price'].values
        qtys = df['quantity'].values

        for i in range(len(times)):
            ts = times[i]
            
            if next_snapshot_time is None:
                next_snapshot_time = ts + INTERVAL_NS

            while ts >= next_snapshot_time:
                b_p, b_q = get_sorted_book(bids, reverse=True)
                a_p, a_q = get_sorted_book(asks, reverse=False)
                
                feat_vec = compute_features_numba(b_p, b_q, a_p, a_q, next_snapshot_time)
                if feat_vec[1] > 0: 
                    all_features.append(feat_vec)
                
                next_snapshot_time += INTERVAL_NS

            p_val = float(prices[i])
            q_val = float(qtys[i])
            target = bids if sides[i] == 'bid' else asks
            
            if q_val == 0:
                target.pop(p_val, None)
            else:
                target[p_val] = q_val

        file_counter += 1
        progress["processed_files"].append(file_path)

        if file_counter % SAVE_INTERVAL_FILES == 0:
            print(f"--- Checkpoint reached at {file_counter} files ---")
            save_progress(progress["processed_files"], next_snapshot_time, all_features)

    if not all_features:
        print("No features generated.")
        return

    cols = [
        'timestamp', 'best_bid', 'best_ask', 'mid_price', 'spread', 'spread_bps', 'micro_price',
        'bid_vol_1', 'bid_vol_5', 'bid_vol_10', 'ask_vol_1', 'ask_vol_5', 'ask_vol_10',
        'obi_1', 'obi_5', 'obi_10', 'bid_density', 'ask_density', 'empty1', 'empty2'
    ]
    
    final_df = pd.DataFrame(np.array(all_features), columns=cols)
    final_df = final_df.drop(columns=['empty1', 'empty2'])

    print("Computing target variables...")
    PERIODS_15_MIN = int((15 * 60 * 1_000_000_000) / INTERVAL_NS)
    final_df['future_mid_price_15m'] = final_df['mid_price'].shift(-PERIODS_15_MIN)
    final_df['target_price_up'] = (final_df['future_mid_price_15m'] > final_df['mid_price']).astype(int)
    final_df = final_df.dropna(subset=['future_mid_price_15m'])

    output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
    final_df.to_parquet(output_path, index=False)
    print(f"Complete! Saved to {output_path}")

if __name__ == "__main__":
    process_orderbook_stream()

SystemError: CPUDispatcher(<function process_chunk_numba at 0x7f176bdcef20>) returned a result with an exception set

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import json
from numba import njit
from datetime import datetime
from sortedcontainers import SortedDict
import concurrent.futures
from typing import Optional
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Configuration ────────────────────────────────────────────────────────────
INPUT_FOLDER        = "orderbook_chunks"
OUTPUT_FOLDER       = "features"
CHECKPOINT_FOLDER   = "checkpoints"
OUTPUT_FILE         = "orderbook_features.parquet"
INTERVAL_NS         = 30 * 1_000_000_000   # 30 seconds in nanoseconds
LEVELS_TO_EXTRACT   = 10
SAVE_INTERVAL_FILES = 15
MAX_WORKERS         = 4                     # parallel parquet readers
PERIODS_15_MIN      = int((15 * 60 * 1_000_000_000) / INTERVAL_NS)

FEATURE_COLS = [
    "timestamp", "best_bid", "best_ask", "mid_price", "spread", "spread_bps",
    "micro_price", "bid_vol_1", "bid_vol_5", "bid_vol_10",
    "ask_vol_1",  "ask_vol_5", "ask_vol_10",
    "obi_1", "obi_5", "obi_10", "bid_density", "ask_density",
]


# ── Numba kernel ─────────────────────────────────────────────────────────────
@njit(cache=True)
def compute_features_numba(
    bid_prices: np.ndarray,
    bid_qtys:   np.ndarray,
    ask_prices: np.ndarray,
    ask_qtys:   np.ndarray,
    timestamp:  float,
) -> np.ndarray:
    """
    Compute 18 order-book features from pre-sorted level arrays.

    Parameters
    ----------
    bid_prices : descending
    ask_prices : ascending
    """
    res = np.zeros(18)

    if len(bid_prices) == 0 or len(ask_prices) == 0:
        return res

    best_bid     = bid_prices[0]
    best_bid_qty = bid_qtys[0]
    best_ask     = ask_prices[0]
    best_ask_qty = ask_qtys[0]

    mid_price  = (best_bid + best_ask) * 0.5
    spread     = best_ask - best_bid
    spread_bps = (spread / mid_price) * 10_000.0 if mid_price > 0.0 else 0.0

    denom      = best_bid_qty + best_ask_qty
    micro_price = (
        (best_bid * best_ask_qty + best_ask * best_bid_qty) / denom
        if denom > 0.0 else mid_price
    )

    res[0] = timestamp
    res[1] = best_bid
    res[2] = best_ask
    res[3] = mid_price
    res[4] = spread
    res[5] = spread_bps
    res[6] = micro_price

    # Aggregated volumes & OBI at 1 / 5 / 10 levels
    ns = (1, 5, 10)
    for i in range(3):
        n     = ns[i]
        b_vol = np.sum(bid_qtys[:n])
        a_vol = np.sum(ask_qtys[:n])
        total = b_vol + a_vol

        res[7  + i] = b_vol
        res[10 + i] = a_vol
        res[13 + i] = (b_vol - a_vol) / total if total > 0.0 else 0.0

    # Price-range densities
    last_bid = min(len(bid_prices) - 1, 9)
    last_ask = min(len(ask_prices) - 1, 9)

    bid_range = bid_prices[0] - bid_prices[last_bid]
    ask_range = ask_prices[last_ask] - ask_prices[0]

    res[16] = res[9]  / bid_range if bid_range > 0.0 else 0.0
    res[17] = res[12] / ask_range if ask_range > 0.0 else 0.0

    return res


# ── Numba warm-up (avoids first-call JIT penalty during real processing) ─────
def _warmup_numba() -> None:
    dummy = np.array([1.0])
    compute_features_numba(dummy, dummy, dummy, dummy, 0.0)


# ── Book helpers ─────────────────────────────────────────────────────────────
def _book_arrays(book: SortedDict, descending: bool) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract up to LEVELS_TO_EXTRACT price/qty arrays from a SortedDict.
    SortedDict keeps keys in ascending order; we slice accordingly.
    """
    n = len(book)
    if n == 0:
        return np.empty(0, dtype=np.float64), np.empty(0, dtype=np.float64)

    take    = min(n, LEVELS_TO_EXTRACT)
    keys_view = book.keys()

    if descending:
        # Bids: highest prices first → take last `take` keys reversed
        prices = np.fromiter(
            reversed(keys_view[-take:]),   # SortedList slice
            dtype=np.float64, count=take
        )
    else:
        # Asks: lowest prices first → take first `take` keys
        prices = np.fromiter(
            keys_view[:take],
            dtype=np.float64, count=take
        )

    qtys = np.fromiter(
        (book[p] for p in prices), dtype=np.float64, count=take
    )
    return prices, qtys


# ── Checkpoint helpers ────────────────────────────────────────────────────────
def load_progress() -> dict:
    path = os.path.join(CHECKPOINT_FOLDER, "metadata.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {"processed_files": [], "next_snapshot_time": None}


def save_progress(
    processed_files:    list[str],
    next_snapshot_time: Optional[int],
    features_acc:       list[np.ndarray],
) -> None:
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)

    meta = {
        "processed_files":   processed_files,
        "next_snapshot_time": int(next_snapshot_time) if next_snapshot_time is not None else None,
    }
    with open(os.path.join(CHECKPOINT_FOLDER, "metadata.json"), "w") as f:
        json.dump(meta, f)

    if features_acc:
        arr = np.array(features_acc, dtype=np.float64)
        df  = pd.DataFrame(arr, columns=FEATURE_COLS)
        df.to_parquet(
            os.path.join(CHECKPOINT_FOLDER, f"partial_{len(processed_files)}.parquet"),
            index=False,
        )
        log.info("Checkpoint saved (%d rows, %d files).", len(df), len(processed_files))


# ── Parallel file reader ──────────────────────────────────────────────────────
def _read_parquet(path: str) -> pd.DataFrame:
    """Read a single parquet file, ensuring correct dtypes."""
    df = pd.read_parquet(path, columns=["received_time", "side", "price", "quantity"])

    if not pd.api.types.is_integer_dtype(df["received_time"]):
        df["received_time"] = df["received_time"].values.astype("int64")

    # Encode side as bool (True = bid) to avoid per-row string comparison
    df["is_bid"] = df["side"] == "bid"
    df.drop(columns=["side"], inplace=True)

    return df


def prefetch_files(
    files: list[str],
    max_workers: int = MAX_WORKERS,
) -> dict[str, pd.DataFrame]:
    """Load all parquet files in parallel."""
    results: dict[str, pd.DataFrame] = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_map = {pool.submit(_read_parquet, f): f for f in files}
        for future in concurrent.futures.as_completed(future_map):
            path = future_map[future]
            try:
                results[path] = future.result()
            except Exception as exc:
                log.error("Failed to read %s: %s", path, exc)
    return results


# ── Snapshot emission ─────────────────────────────────────────────────────────
def _emit_snapshots(
    bids:              SortedDict,
    asks:              SortedDict,
    next_snapshot_time: int,
    up_to_ts:          int,
    all_features:      list[np.ndarray],
) -> int:
    """
    Emit all 30-second snapshots whose boundary ≤ up_to_ts.
    Returns updated next_snapshot_time.
    """
    while up_to_ts >= next_snapshot_time:
        b_p, b_q = _book_arrays(bids, descending=True)
        a_p, a_q = _book_arrays(asks, descending=False)

        feat = compute_features_numba(b_p, b_q, a_p, a_q, float(next_snapshot_time))

        if feat[1] > 0.0:          # best_bid must be valid
            all_features.append(feat)

        next_snapshot_time += INTERVAL_NS

    return next_snapshot_time


# ── Main processing loop ──────────────────────────────────────────────────────
def process_orderbook_stream() -> None:
    os.makedirs(OUTPUT_FOLDER,      exist_ok=True)
    os.makedirs(CHECKPOINT_FOLDER,  exist_ok=True)

    _warmup_numba()

    progress           = load_progress()
    processed_set      = set(progress["processed_files"])
    next_snapshot_time: Optional[int] = progress["next_snapshot_time"]

    # SortedDict keeps keys sorted → no manual sort needed
    bids: SortedDict = SortedDict()
    asks: SortedDict = SortedDict()

    file_pattern = os.path.join(INPUT_FOLDER, "*.parquet")
    all_files    = sorted(glob.glob(file_pattern))
    todo_files   = [f for f in all_files if f not in processed_set]

    if not todo_files:
        log.info("No new input files found.")
        return

    log.info("Total files: %d  |  To process: %d", len(all_files), len(todo_files))

    # Pre-fetch all files in parallel (memory permitting; swap for a sliding
    # window if RAM is tight)
    log.info("Pre-fetching %d files with %d workers…", len(todo_files), MAX_WORKERS)
    cache = prefetch_files(todo_files, max_workers=MAX_WORKERS)

    all_features:    list[np.ndarray] = []
    file_counter:    int              = 0
    processed_files: list[str]        = list(progress["processed_files"])

    for file_path in todo_files:
        df = cache.get(file_path)
        if df is None:
            log.warning("Skipping %s (read error).", file_path)
            continue

        log.info("Processing: %s  (%d rows)", os.path.basename(file_path), len(df))

        # ── Convert columns to plain NumPy arrays for fast iteration ─────────
        times:   np.ndarray = df["received_time"].values   # int64
        is_bids: np.ndarray = df["is_bid"].values          # bool
        prices:  np.ndarray = df["price"].values.astype(np.float64)
        qtys:    np.ndarray = df["quantity"].values.astype(np.float64)

        # ── Initialise snapshot clock on first event seen ─────────────────────
        if next_snapshot_time is None:
            next_snapshot_time = int(times[0]) + INTERVAL_NS

        # ── Row-level update loop (unavoidably sequential) ───────────────────
        # Using enumerate + zip over pre-extracted arrays is the fastest pure-
        # Python approach; Cython / Rust extension would be the next step.
        for ts, is_bid, price, qty in zip(times, is_bids, prices, qtys):
            ts = int(ts)

            # Emit any snapshots that have passed
            if ts >= next_snapshot_time:
                next_snapshot_time = _emit_snapshots(
                    bids, asks, next_snapshot_time, ts, all_features
                )

            # Apply book update
            target = bids if is_bid else asks
            if qty == 0.0:
                target.pop(price, None)
            else:
                target[price] = qty

        # ── Checkpoint ───────────────────────────────────────────────────────
        file_counter += 1
        processed_files.append(file_path)

        if file_counter % SAVE_INTERVAL_FILES == 0:
            log.info("--- Checkpoint at file %d ---", file_counter)
            save_progress(processed_files, next_snapshot_time, all_features)

        # Free memory from cache entry
        del cache[file_path]

    # ── Assemble final DataFrame ──────────────────────────────────────────────
    if not all_features:
        log.info("No features generated.")
        return

    log.info("Assembling final DataFrame (%d snapshots)…", len(all_features))
    arr      = np.array(all_features, dtype=np.float64)
    final_df = pd.DataFrame(arr, columns=FEATURE_COLS)

    # ── Target variables ──────────────────────────────────────────────────────
    log.info("Computing target variables (look-ahead = %d periods)…", PERIODS_15_MIN)
    final_df["future_mid_price_15m"] = final_df["mid_price"].shift(-PERIODS_15_MIN)
    final_df["target_price_up"] = (
        final_df["future_mid_price_15m"] > final_df["mid_price"]
    ).astype(np.int8)
    final_df.dropna(subset=["future_mid_price_15m"], inplace=True)

    output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
    final_df.to_parquet(output_path, index=False)
    log.info("Done! Saved %d rows → %s", len(final_df), output_path)


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    process_orderbook_stream()

# Claude 2

In [ ]:
import os
import glob
import polars as pl
import numpy as np
import json
from numba import njit
from numba.typed import Dict
from numba.core import types
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
INPUT_FOLDER        = "orderbook_chunks"
OUTPUT_FOLDER       = "features"
CHECKPOINT_FOLDER   = "checkpoints"
OUTPUT_FILE         = "orderbook_features.parquet"
INTERVAL_NS         = np.int64(30 * 1_000_000_000)
SAVE_INTERVAL_FILES = 15

COLS = [
    "timestamp", "best_bid", "best_ask", "mid_price", "spread", "spread_bps", "micro_price",
    "bid_vol_1", "bid_vol_5", "bid_vol_10", "ask_vol_1", "ask_vol_5", "ask_vol_10",
    "obi_1", "obi_5", "obi_10", "bid_density", "ask_density",
]

# ── Numba kernels ─────────────────────────────────────────────────────────────
@njit(cache=True)   # cache=True: compiled bytecode is saved to disk — no recompile on next run
def _compute_features(bid_prices, bid_qtys, ask_prices, ask_qtys, timestamp):
    if len(bid_prices) == 0 or len(ask_prices) == 0:
        return np.zeros(18)

    best_bid     = bid_prices[0]
    best_bid_qty = bid_qtys[0]
    best_ask     = ask_prices[0]
    best_ask_qty = ask_qtys[0]

    mid_price   = (best_bid + best_ask) * 0.5
    spread      = best_ask - best_bid
    spread_bps  = (spread / mid_price) * 10_000.0 if mid_price > 0.0 else 0.0
    denom       = best_bid_qty + best_ask_qty
    micro_price = (best_bid * best_ask_qty + best_ask * best_bid_qty) / denom if denom > 0.0 else mid_price

    res    = np.zeros(18)
    res[0] = timestamp
    res[1] = best_bid
    res[2] = best_ask
    res[3] = mid_price
    res[4] = spread
    res[5] = spread_bps
    res[6] = micro_price

    # OBI and volume at 1 / 5 / 10 levels — plain loops so Numba can unroll
    for i, n in enumerate((1, 5, 10)):
        b_vol = 0.0
        a_vol = 0.0
        for j in range(min(n, len(bid_prices))):
            b_vol += bid_qtys[j]
        for j in range(min(n, len(ask_prices))):
            a_vol += ask_qtys[j]
        res[7  + i] = b_vol
        res[10 + i] = a_vol
        total = b_vol + a_vol
        res[13 + i] = (b_vol - a_vol) / total if total > 0.0 else 0.0

    bid_range = bid_prices[0] - bid_prices[min(len(bid_prices) - 1, 9)]
    ask_range = ask_prices[min(len(ask_prices) - 1, 9)] - ask_prices[0]
    res[16] = res[9]  / bid_range if bid_range > 0.0 else 0.0
    res[17] = res[12] / ask_range if ask_range > 0.0 else 0.0
    return res


@njit(cache=True)
def process_events(times, sides_bool, prices, qtys,
                   bids, asks, next_snapshot_time, interval_ns):
    """
    KEY OPTIMIZATION: entire hot loop lives here in Numba — zero Python overhead
    per row.  Typed dicts are passed in and mutated in-place so state persists
    across files without any serialization cost.

    Returns (feature_matrix[N, 18], updated_next_snapshot_time).
    """
    n = len(times)
    if n == 0:
        return np.zeros((0, 18)), next_snapshot_time

    # Pre-allocate generously; trim at the end.
    time_span = times[n - 1] - times[0]
    max_snaps = int(time_span // interval_ns) + 1_000
    features  = np.zeros((max_snaps, 18))
    feat_count = 0

    for i in range(n):
        ts = times[i]

        if next_snapshot_time == np.int64(-1):
            next_snapshot_time = ts + interval_ns

        # ── emit one snapshot per elapsed interval ────────────────────────────
        while ts >= next_snapshot_time:
            nb = len(bids)
            na = len(asks)
            if nb > 0 and na > 0:
                bp = np.empty(nb)
                bq = np.empty(nb)
                ap = np.empty(na)
                aq = np.empty(na)
                j  = 0
                for k, v in bids.items():
                    bp[j] = k
                    bq[j] = v
                    j += 1
                j = 0
                for k, v in asks.items():
                    ap[j] = k
                    aq[j] = v
                    j += 1

                # np.argsort(-x) is safer than [::-1] inside Numba
                bid_ord = np.argsort(-bp)
                ask_ord = np.argsort(ap)

                feat = _compute_features(
                    bp[bid_ord], bq[bid_ord],
                    ap[ask_ord], aq[ask_ord],
                    next_snapshot_time,
                )
                if feat[1] > 0.0 and feat_count < max_snaps:
                    features[feat_count] = feat
                    feat_count += 1

            next_snapshot_time += interval_ns

        # ── apply delta update ────────────────────────────────────────────────
        p = prices[i]
        q = qtys[i]
        if sides_bool[i]:           # True → bid
            if q == 0.0:
                if p in bids:
                    del bids[p]
            else:
                bids[p] = q
        else:                       # False → ask
            if q == 0.0:
                if p in asks:
                    del asks[p]
            else:
                asks[p] = q

    return features[:feat_count], next_snapshot_time


# ── Checkpoint helpers ────────────────────────────────────────────────────────
def load_progress():
    path = os.path.join(CHECKPOINT_FOLDER, "metadata.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {"processed_files": [], "next_snapshot_time": None}


def save_progress(processed_files, next_snapshot_time, feature_chunks):
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)
    with open(os.path.join(CHECKPOINT_FOLDER, "metadata.json"), "w") as f:
        json.dump({
            "processed_files": processed_files,
            "next_snapshot_time": int(next_snapshot_time) if next_snapshot_time != -1 else None,
        }, f)
    if feature_chunks:
        pl.DataFrame(np.vstack(feature_chunks), schema=COLS).write_parquet(
            os.path.join(CHECKPOINT_FOLDER, f"partial_{len(processed_files)}.parquet")
        )


# ── Main pipeline ─────────────────────────────────────────────────────────────
def process_orderbook_stream():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)

    progress      = load_progress()
    processed_set = set(progress["processed_files"])
    raw_ts        = progress["next_snapshot_time"]
    next_snap_ts  = np.int64(raw_ts) if raw_ts is not None else np.int64(-1)

    # Typed dicts are kept alive across files — no serialization, no Python dict copies
    bids = Dict.empty(key_type=types.float64, value_type=types.float64)
    asks = Dict.empty(key_type=types.float64, value_type=types.float64)

    files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.parquet")))
    if not files:
        print("No input files found.")
        return

    # ── JIT warm-up — pay compile cost once, not on your first real file ──────
    print("Warming up Numba JIT …")
    _wb, _wa = (Dict.empty(key_type=types.float64, value_type=types.float64) for _ in range(2))
    _wb[100.0] = 1.0
    _wa[101.0] = 1.0
    process_events(
        np.array([0], dtype=np.int64), np.array([True]),
        np.array([100.0]), np.array([0.5]),
        _wb, _wa, np.int64(-1), INTERVAL_NS,
    )
    print("JIT ready.\n")

    all_features = []
    file_counter = 0

    for file_path in files:
        if file_path in processed_set:
            continue

        t0 = datetime.now()

        # Polars lazy scan: reads only the columns we need, ~3-5× faster than pd.read_parquet
        df = (
            pl.scan_parquet(file_path)
            .select(["received_time", "side", "price", "quantity"])
            .collect()
        )

        times      = df["received_time"].cast(pl.Int64).to_numpy()
        sides_bool = (df["side"] == "bid").to_numpy()          # bool, no copy
        prices     = df["price"].cast(pl.Float64).to_numpy()
        qtys       = df["quantity"].cast(pl.Float64).to_numpy()

        feat_chunk, next_snap_ts = process_events(
            times, sides_bool, prices, qtys,
            bids, asks, next_snap_ts, INTERVAL_NS,
        )

        elapsed = (datetime.now() - t0).total_seconds()
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {os.path.basename(file_path)}"
              f"  →  {len(feat_chunk):>4} snapshots  ({len(times):,} rows in {elapsed:.2f}s)")

        if len(feat_chunk) > 0:
            all_features.append(feat_chunk)

        file_counter += 1
        progress["processed_files"].append(file_path)

        if file_counter % SAVE_INTERVAL_FILES == 0:
            print(f"  ↳ Checkpoint after {file_counter} files")
            save_progress(progress["processed_files"], next_snap_ts, all_features)

    if not all_features:
        print("No features generated.")
        return

    print("\nAssembling final dataset …")
    final_df = pl.DataFrame(np.vstack(all_features), schema=COLS)

    PERIODS_15M = int((15 * 60 * 1_000_000_000) // INTERVAL_NS)   # = 30
    final_df = (
        final_df
        .with_columns(pl.col("mid_price").shift(-PERIODS_15M).alias("future_mid_15m"))
        .with_columns(
            (pl.col("future_mid_15m") > pl.col("mid_price")).cast(pl.Int8).alias("target_price_up")
        )
        .drop_nulls("future_mid_15m")
    )

    out_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
    final_df.write_parquet(out_path)
    print(f"Done! {len(final_df):,} rows → {out_path}")


if __name__ == "__main__":
    process_orderbook_stream()

Warming up Numba JIT …
JIT ready.

[10:17:42] 2025-10-06_BTCUSDT.parquet_chunk_01.parquet  →   480 snapshots  (21,306,044 rows in 3.39s)
[10:17:45] 2025-10-06_BTCUSDT.parquet_chunk_02.parquet  →   480 snapshots  (15,937,766 rows in 3.02s)
[10:17:48] 2025-10-06_BTCUSDT.parquet_chunk_03.parquet  →   480 snapshots  (15,582,868 rows in 3.26s)
[10:17:53] 2025-10-06_BTCUSDT.parquet_chunk_04.parquet  →   480 snapshots  (25,809,391 rows in 4.72s)
[10:17:57] 2025-10-06_BTCUSDT.parquet_chunk_05.parquet  →   480 snapshots  (21,210,245 rows in 4.35s)
[10:18:01] 2025-10-06_BTCUSDT.parquet_chunk_06.parquet  →   480 snapshots  (14,593,570 rows in 3.67s)
[10:18:04] 2025-10-07_BTCUSDT.parquet_chunk_01.parquet  →   481 snapshots  (15,373,246 rows in 3.79s)
[10:18:08] 2025-10-07_BTCUSDT.parquet_chunk_02.parquet  →   480 snapshots  (15,428,553 rows in 3.84s)
[10:18:12] 2025-10-07_BTCUSDT.parquet_chunk_03.parquet  →   480 snapshots  (15,902,336 rows in 3.89s)
